# SupplementRx AI — Bounded Preprocessing Pilot

This notebook **implements and runs** the first bounded pilot against contract 0.1.0. It is separate from the audit/design notebooks. Raw data and `read.md` stay unchanged. Processed outputs are versioned under `data/processed/poc/<build_id>/`.

## Broad steps

1. Load the pinned contract and selected raw inputs; run synthetic and real-data validator fixtures.
2. Extract source-scoped provenance, occurrences, servings, assertions, vocabulary evidence, lookups, references and topic associations.
3. Apply the allowed name-search and plain-mass representation rules; retain original values, missingness, qualifiers and serving context.
4. Validate output structure, reference integrity, source multiplicity, evidence support and preserved review states.
5. Inspect positive, blocked and ambiguous results, then verify a complete rerun produces the same build and bytes.

**Boundaries:** seven DSLD labels, one pending NatMed claim with its monograph/reference documents, three Reddit comments, associated saved terminology lookup candidates and the MeSH hierarchy fixture. No network requests, canonical identity assignment, clinical approval, semantic merging, graph export, embeddings or training. Existing scope ambiguities remain in the review queue.

Use `supplementsrx_env` as the Python environment. The implementation is in `../scripts/preprocess_pilot.py`, fixtures in `../tests/test_preprocess_pilot.py`, and the contract in `../schemas/poc_preprocessing_contract_v0.1.json`. The contract remains the historical design baseline; this notebook and the output manifest record implementation progress.

**Output interpretation:** all 21 logical table files are emitted, with intentionally empty/deferred tables explicitly labeled in the manifest. Snapshots are container-bound, so file changes may conservatively version multiple records. Reference availability means availability in the selected pilot output, not nonexistence remotely or absence from every archived raw file. The stored source key and reason preserve that limitation.


In [1]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd

pilot_project = Path.cwd()
if not (pilot_project / "scripts").is_dir():
    pilot_project = pilot_project.parent
assert (pilot_project / "schemas/poc_preprocessing_contract_v0.1.json").is_file()
if str(pilot_project / "scripts") not in sys.path:
    sys.path.insert(0, str(pilot_project / "scripts"))
from preprocess_pilot import run, encode

pilot_output, pilot_validation, pilot = run(pilot_project)
print("Build:", pilot_output.relative_to(pilot_project))
print("Fixture results:", pilot_validation["fixtures"])
print("Source inventory:", pilot_validation["source_inventory"])
print("Evidence values verified:", pilot_validation["evidence_checks"])
print("Tables:")
print(pd.Series(pilot_validation["row_counts"]).to_string())
print("Normalization outcomes:", pilot_validation["normalization_outcomes"])
print("No approvals or semantic merges:", {k:pilot_validation[k] for k in ["accepted_mappings","clinical_approvals","semantic_merges"]})


Build: data/processed/poc/pilot-v1-a88f090b41e160766e4a
Fixture results: {'tests_run': 26, 'failures': 0, 'errors': 0, 'skipped': 0}
Source inventory: {'dsld_recursive_rows': 16, 'dsld_forms': 11, 'dsld_other_occurrences': 48, 'dsld_quantity_entries': 16}
Evidence values verified: 3730
Tables:
artifacts                  13
source_records            149
snapshots                 149
capture_events             16
evidence                 3730
occurrences                76
servings                    7
assertions                104
term_values              2212
vocabulary_statements    1205
lookups                    63
lookup_origins             81
mapping_assertions         40
source_references         531
topic_associations          6
passages                    0
transformations           198
review_queue              481
review_events               0
canonical_identities        0
comparison_groups           0
Normalization outcomes: {'unchanged': 109, 'normalized': 80, 'blocked': 9}


## Quantity inspection: equivalent units, unknowns and a source conflict

Each mass-comparison row below remains attached to its original assertion. `original_quantity` is retained as source evidence; the normalized amount is a separate exact decimal string. An equal comparison value never approves a common product/variant. Parent amounts are not assigned to forms/components.

The BetterYou conflict is supplied by the pinned quantity audit and checked against the saved label. Its numeric/text serving disagreement blocks dependent mass comparison in this conservative pilot. NP quantities remain unknown rather than becoming zero doses. Units outside the explicit mass allowlist are retained without conversion.

`<unknown>` is a display marker only; stored missing/blocked values remain JSON null with their reason codes.


In [2]:
quantity_preview = []
for t in pilot.tables["transformations"].values():
    if t["destination_field"] != "mass_comparison":
        continue
    assertion = pilot.tables["assertions"][t["owner_id"]]
    snapshot = pilot.tables["snapshots"][assertion["snapshot_id"]]
    source = pilot.tables["source_records"][snapshot["record_id"]]
    q = assertion["original_fields"]["quantity"]
    nv = t["normalized_value"]
    quantity_preview.append({
        "label": source["native_id"], "ingredient": assertion["original_fields"]["ingredient_name"],
        "original_quantity": str(q.get("quantity")), "unit": q.get("unit"),
        "basis": str(q.get("servingSizeQuantity")) + " " + str(q.get("servingSizeUnit")),
        "comparison_mg": nv["amount"] if nv else None,
        "operator": q.get("operator"), "outcome": t["outcome"],
        "reason": "; ".join(t["reason_codes"]), "assertion_id": assertion["id"]})
pilot_quantities = pd.DataFrame(quantity_preview).sort_values(["label", "ingredient"])
print(pilot_quantities.drop(columns=["assertion_id"]).fillna("<unknown>").to_string(index=False))
print("\nBPI check: comparison values agree, labels remain distinct:")
print(pilot_quantities[pilot_quantities["label"].isin(["25731","43261"])][["label","ingredient","original_quantity","unit","basis","comparison_mg"]].to_string(index=False))


 label                                               ingredient original_quantity    unit        basis comparison_mg operator    outcome                       reason
  1452                                                Magnesium               100      mg 1 Capsule(s)           100        = normalized                             
249674                                               Acacia Gum                 0      NP 1 Capsule(s)     <unknown>        =    blocked    unsupported_source_amount
249674                  Bifidobacterium animalis lactis (Bl-07)                 0      NP 1 Capsule(s)     <unknown>        =    blocked    unsupported_source_amount
249674                           Bifidobacterium lactis (Bl-04)                 0      NP 1 Capsule(s)     <unknown>        =    blocked    unsupported_source_amount
249674                                   Fructooligosaccharides                 0      NP 1 Capsule(s)     <unknown>        =    blocked    unsupported_source_amount
2496

## Mapping, clinical and conversation context

Terminology candidate links remain `candidate_unspecified` / `needs_review`; no source query result is accepted automatically. MeSH lookup provenance includes the original term→concept→record resolution bindings and candidate index, in addition to the saved RDF definitions.

The clinical claim's source excerpt and field-support span are checked against the original monograph section. Structured null dose/route fields stay null even though numbers appear in the excerpt. Reddit topic associations retain `thread_context` or `own_text` at the **record level**; neither is an approval or an automatic per-topic use statement.


In [3]:
clinical = [a for a in pilot.tables["assertions"].values() if a["kind"] == "clinical"]
print("Clinical assertions:", len(clinical))
for a in clinical:
    print("Native claim:", a["native_assertion_id"])
    print("Review:", a["source_review"])
    print("Dose context:", a["original_fields"]["dose"])
    print("Source support evidence IDs:", a["evidence_ids"])
print("\nCandidate mapping states:")
print(pd.DataFrame(list(pilot.tables["mapping_assertions"].values())).groupby(["relation","review_state"]).size().to_string())
reference_preview = []
for r in pilot.tables["source_references"].values():
    if r["role"] not in ["REPLY_TO","IN_THREAD"]:
        continue
    record = pilot.tables["source_records"][pilot.tables["snapshots"][r["source_snapshot_id"]]["record_id"]]
    reference_preview.append({"comment":record["native_id"],"role":r["role"],"original_target":r["original_reference"],
        "identifier_state":r["identifier_state"],"availability":r["target_availability"],"resolution":r["resolution_state"]})
print("\nReddit references (availability within selected pilot):")
print(pd.DataFrame(reference_preview).fillna("<unknown>").to_string(index=False))
print("\nTopic association evidence:")
for t in pilot.tables["topic_associations"].values():
    record = pilot.tables["source_records"][pilot.tables["snapshots"][t["snapshot_id"]]["record_id"]]
    print(record["native_id"], t["original_topic"], t["mention_scope"], "scope level:", t["scope_level"])
print("\nReview queue by stage (issue rows; overlapping owners):")
print(pd.Series(pilot_validation["review_queue_by_stage"]).to_string())


Clinical assertions: 1
Native claim: claim:3a1e7c59e8144dd11cfe90e000be0afa605130adb7d5baf9a94e590a529ea45f
Review: {'reviewed_at': None, 'reviewer': None, 'state': 'pending'}
Dose context: {'duration': None, 'frequency': None, 'maximum': None, 'minimum': None, 'qualifier': None, 'regimen': None, 'route': None, 'text': None, 'unit': None, 'value': None}
Source support evidence IDs: ['evidence:5ae005fc94f57d975b614ed61ec9dac85fc6438024f38c33de2fea19b1ac0a00', 'evidence:a13ab9e2860b4373818390172c4a292bf63dd581ead630c7b4178f3cbdb388fe', 'evidence:6735c8dae05985f4c07d8edd7c27a5e418dbcc78550e86eae431b570a0a077da']

Candidate mapping states:
relation               review_state
candidate_unspecified  needs_review    40

Reddit references (availability within selected pilot):
         comment      role original_target identifier_state   availability resolution
t1_19rg1ztatzi8z  REPLY_TO      t3_1xbme8k           usable  not_collected unresolved
t1_19rl50ku815cw IN_THREAD      t3_1xe6ara       

## Replay and preservation check

The next cell reruns the complete bounded pilot, including its fixtures. It verifies the same build directory, identical table/manifest/report bytes, unchanged table counts and preserved raw/prior-artifact hashes. These checks demonstrate offline replay behavior; they do not simulate all possible publisher revisions or validate full graph/clinical behavior.


In [4]:
first_build_hashes = {f.name: hashlib.sha256(f.read_bytes()).hexdigest() for f in pilot_output.iterdir() if f.is_file()}
replayed_output, replayed_validation, replayed_pilot = run(pilot_project)
second_build_hashes = {f.name: hashlib.sha256(f.read_bytes()).hexdigest() for f in replayed_output.iterdir() if f.is_file()}
assert replayed_output == pilot_output
assert first_build_hashes == second_build_hashes
assert replayed_validation == pilot_validation
assert pilot.protected_hashes() == pilot.protected
print("Identical rerun verified:", len(first_build_hashes), "output files")
print("Same logical row counts; no duplicate rows appended.")
print("Raw data, read.md and earlier audit/review artifacts unchanged.")
print("Bounded fixture tests:", pilot_validation["fixtures"]["tests_run"])
print("V8: examples displayed and inspected; no automatic scope expansion.")


Identical rerun verified: 24 output files
Same logical row counts; no duplicate rows appended.
Raw data, read.md and earlier audit/review artifacts unchanged.
Bounded fixture tests: 26
V8: examples displayed and inspected; no automatic scope expansion.


## Pilot outcome and remaining work

The pilot now produces real processed, source-scoped data. Inspection confirms that BPI's 1.5 g and 1,500 mg representations yield the same mass comparison while both labels and their composition evidence remain separate. Unknown NP amounts and the BetterYou conflict remain blocked. No form inherits a parent quantity. The NatMed claim stays pending with null structured dose fields; Reddit topic/reference roles remain distinct.

All focused fixtures passed, the selected source evidence was checked, and the complete rerun reproduced identical output bytes. The broader repository test suite was also run after implementation. Counts and hashes are recorded in the executed cells and build manifest rather than hard-coded here.

**Limitations:** this is seven DSLD labels and associated evidence, not normalization of all 400 labels or all collected datasets. Missing/uncollected target references and candidate mappings stay unresolved. Availability is measured within the selected pilot; some targets may exist elsewhere in the archive. Citation extraction, passage projection, semantic identity review, product grouping, clinical approval, deduplication beyond replay handling, graph export and embeddings remain deferred. Container-bound snapshots are intentionally conservative.

**Next:** review these actual outputs and unresolved issues before extending the same validated source-scoped extraction to a larger sample. The schema contract remains versioned; any widened transform or identity acceptance needs its own reviewed rule and tests. No GitHub push or read.md update is performed by this notebook.


## Evidence review of all 40 pilot candidate mappings

This bounded review covers 17 source occurrences and 12 ingredient names: **24 supported proposals, 8 evidence gaps, 8 incompatible exact-match candidates**. The supported proposals comprise 18 `denotes_concept` and 6 `exact_material_match` relationships. These are recommendations, not accepted decisions: A6 remains pending. No raw data, original pilot mappings, canonical identities or clinical approval states are changed.

The counts refer to candidate links, not unique supplements or products. This is not an exhaustive external candidate search. Full review JSON includes source context, target properties, saved aliases and scope statements, evidence IDs and artifact hashes. Reproduce from the project root using `supplementsrx_env/bin/python scripts/review_pilot_candidates.py`.


In [5]:
from pathlib import Path
import json, hashlib
import pandas as pd
review_root = Path.cwd()
if not (review_root / 'data').is_dir():
    review_root = review_root.parent
candidate_review_dir = review_root / 'data/processed/identity/pilot-candidate-review-v1-f03a6ba885c074ba'
candidate_review_bytes = (candidate_review_dir / 'review.json').read_bytes()
assert hashlib.sha256(candidate_review_bytes).hexdigest() == 'f03a6ba885c074ba57951710fe0ba82d623507c6360225a4201c4b9015c92f04'
candidate_review = json.loads(candidate_review_bytes)
assert len(candidate_review['reviews']) == 40
assert all(not r['approval_applied'] for r in candidate_review['reviews'])
print(pd.DataFrame(candidate_review['groups']).to_string(index=False))
print(json.dumps(json.loads((candidate_review_dir / 'validation.json').read_text()), indent=2))


             ingredient       labels  candidates  supported_for_review  needs_evidence  incompatible_exact_match
             Acacia Gum       249674           2                     1               0                         1
              Alkaloids 74939, 74940           2                     2               0                         0
               Creatine 25731, 43261           6                     4               2                         0
           Creatine HCl 25731, 43261           6                     2               0                         4
   Creatine Monohydrate 25731, 43261           4                     4               0                         0
Ferric Ammonium Citrate       257287           3                     2               1                         0
  Ferric Sodium Edetate       257287           2                     0               2                         0
 Fructooligosaccharides       249674           2                     1               0          

### Candidate-by-candidate findings, grouped by ingredient

Concept mapping does not establish exact product material. The six exact named-form proposals concern explicitly declared creatine hydrochloride/monohydrate, not whole-product or branded buffering-technology equivalence. Incompatible exact-match recommendations do not rule out separately supported relationships.


In [6]:
for ingredient in sorted({r['ingredient'] for r in candidate_review['reviews']}):
    print('\n' + ingredient)
    for row in candidate_review['reviews']:
        if row['ingredient'] != ingredient:
            continue
        print(f"  Label {row['dsld_label_id']} -> {row['target_source']} {row['target_id']} ({row['target_name']})")
        print(f"    {row['outcome']}; proposed relation: {row['proposed_relation'] or '<unresolved>'}")
        print('    Reason: ' + row['rationale'])
        print('    Next: ' + row['next_action'])



Acacia Gum
  Label 249674 -> mesh http://id.nlm.nih.gov/mesh/D006170 (Gum Arabic)
    supported_for_review; proposed relation: denotes_concept
    Reason: The saved Gum Arabic scope describes exudate from Acacia species, consistent with Acacia Gum. Species and processing remain unknown; no exact botanical material identity is asserted.
    Next: Record an explicit reviewer decision before activating this relation; retain source occurrence and qualifiers.
  Label 249674 -> rxnorm 905301 (arabic gum allergenic extract)
    incompatible_exact_match; proposed relation: <unresolved>
    Reason: The target explicitly denotes arabic gum allergenic extract. The source declares Acacia Gum, with no allergenic-extract preparation qualifier.
    Next: Recommend excluding this candidate from exact identity matching; any alternative relation requires separate evidence and review.

Alkaloids
  Label 74939 -> mesh http://id.nlm.nih.gov/mesh/D000470 (Alkaloids)
    supported_for_review; proposed relat

### Interpretation and remaining decisions

- **Standardization markers:** Withanolides (0.5%) and Alkaloids (0.8%) qualify the Ashwagandha blend. Although stored under DSLD forms, they represent marker classes. Their MeSH class proposals preserve that distinction. The single PubChem structure returned for Withanolides does not identify the whole class.
- **Incompatible exact matches:** Creatine HCl to creatine/creatine monohydrate; Acacia Gum to allergenic extract; fructooligosaccharides to beta-D-fructose; and Withanolides to a specific compound.
- **Evidence gaps:** Generic declarations versus precise PubChem structures, citrate composition/hydration, and ferric sodium edetate scope remain unresolved. MeSH Fe(III)-EDTA includes several salt concepts; a sodium alias alone cannot establish exact identity with the whole record.
- **Separate quantity conflict:** BetterYou's 52 mL versus 0.52 mL serving conflict remains blocked. Identity review does not resolve that conflict or establish a dose.

Next, explicitly review and record accepted/rejected relation decisions with their evidence and scope. Keep unresolved links inactive. Accepted decisions can then feed canonical identity construction; clinical approval remains a separate process.


## Simple mapping confidence — pilot rules v1

One overall confidence label is assigned to each reviewed candidate **for its assessed relationship**. These are rule-based labels, not probabilities or clinical evidence ratings.

| Rule | Confidence | Criterion | Proposed action |
|---|---|---|---|
| C1 | High | Review supports the explicit relationship with compatible form/scope | Eligible for use at that relationship level |
| C2 | Medium | Qualifiers or specificity remain unresolved | Pending review |
| C3 | Low | Form, preparation or entity scope conflicts with exact identity | Exclude from exact matching |

Matching names alone cannot override conflicting scope. The C1 translation uses the previous evidence review; it does not treat every exact string match as High. `denotes_concept` stays a concept mapping, never an exact-material assertion. For Low candidates, `assessed_relation` identifies the failed exact-match test; `proposed_relation` remains null. Medium relationships remain unresolved.

This section produces eligibility recommendations for inspection, not activated KG links or accepted review decisions. Raw data, previous reviews and original pilot mappings remain unchanged. Reproduce from the project root with `supplementsrx_env/bin/python scripts/assign_mapping_confidence.py`.


In [7]:
from pathlib import Path
import json, hashlib
import pandas as pd
confidence_root = Path.cwd()
if not (confidence_root / 'data').is_dir():
    confidence_root = confidence_root.parent
confidence_dir = confidence_root / 'data/processed/identity/mapping-confidence-v1-7c8a0e7ea6f02e69'
confidence_bytes = (confidence_dir / 'confidence.json').read_bytes()
assert hashlib.sha256(confidence_bytes).hexdigest() == '7c8a0e7ea6f02e6987a1483566bd484be411b4441b1d47b2e2259458dfee847f'
confidence_report = json.loads(confidence_bytes)
assert hashlib.sha256((confidence_root / confidence_report['input_review']).read_bytes()).hexdigest() == confidence_report['input_sha256']
print('Overall:', confidence_report['counts'])
print(pd.DataFrame(confidence_report['by_source']).to_string(index=False))
confidence_rows = pd.DataFrame(confidence_report['mappings'])
print('\nHigh-confidence relationship types:')
print(confidence_rows.loc[confidence_rows.confidence == 'High', 'proposed_relation'].value_counts().to_string())
print('\nValidation:', confidence_report['validation'])


Overall: {'High': 24, 'Low': 8, 'Medium': 8}
 source  High  Medium  Low
   mesh    11       1    0
pubchem     4       7    3
 rxnorm     9       0    5

High-confidence relationship types:
proposed_relation
denotes_concept         18
exact_material_match     6

Validation: {'complete_coverage': True, 'relations_preserved': True, 'invalid_states_rejected': True, 'approvals_applied': 0, 'raw_pilot_and_review_unchanged': True}


In [8]:
# Display every candidate, preserving each source occurrence and label.
columns = ['dsld_label_id', 'target_source', 'target_name', 'assessed_relation',
           'proposed_relation', 'confidence', 'rule_id', 'action']
for ingredient, group in confidence_rows.groupby('ingredient', sort=True):
    print('\n' + ingredient)
    print(group[columns].fillna('<unresolved>').to_string(index=False))
    for _, row in group.iterrows():
        print(f"  Label {row['dsld_label_id']} / {row['target_source']} {row['target_id']} [{row['rule_id']}]: {row['reason']}")



Acacia Gum
dsld_label_id target_source                   target_name    assessed_relation proposed_relation confidence rule_id              action
       249674          mesh                    Gum Arabic      denotes_concept   denotes_concept       High      C1    eligible_for_use
       249674        rxnorm arabic gum allergenic extract exact_material_match      <unresolved>        Low      C3 exclude_exact_match
  Label 249674 / mesh http://id.nlm.nih.gov/mesh/D006170 [C1]: The saved Gum Arabic scope describes exudate from Acacia species, consistent with Acacia Gum. Species and processing remain unknown; no exact botanical material identity is asserted.
  Label 249674 / rxnorm 905301 [C3]: The target explicitly denotes arabic gum allergenic extract. The source declares Acacia Gum, with no allergenic-extract preparation qualifier.

Alkaloids
dsld_label_id target_source target_name assessed_relation proposed_relation confidence rule_id           action
        74939          mesh   A

### Confidence outcome

**24 High, 8 Medium, 8 Low.** High consists of 18 concept mappings and 6 exact named-form mappings. MeSH: 11 High / 1 Medium / 0 Low; RxNorm: 9 High / 0 Medium / 5 Low; PubChem: 4 High / 7 Medium / 3 Low.

These counts describe this pilot candidate set, not measured precision/recall or overall source quality. The saved CSV and JSON contain the reason, rule, action and original mapping ID for every candidate; JSON also links the previous review and its evidence IDs.

Next: inspect the proposed eligibility decisions before using them in KG construction. Keep Medium candidates pending and Low candidates excluded from exact identity; retain all original source records. The separate serving-size conflict remains unresolved.


## First KG-ready pilot export

The user-authorized export activates the 24 High-confidence mapping proposals **in this local pilot graph only**: 18 `DENOTES_CONCEPT`, 6 `EXACT_MATERIAL_MATCH`. The earlier review artifacts remain historical recommendations. No independent clinical approval is implied.

Scope: seven DSLD label snapshots, all their 75 ingredient/form occurrences, seven servings, 103 label assertions, and 17 selected terminology/compound targets. NatMed and Reddit are not included in this first DSLD-to-terminology graph. Brands and canonical product/ingredient entities are not inferred: original label attributes remain available in `properties_json`.

Node CSVs have `id`, `type`, and `properties_json`; the relationship CSV has explicit endpoint IDs, relationship type, confidence, mapping ID and evidence IDs. These are portable graph tables, not yet an executed Neo4j import. Source label snapshots and occurrences remain distinct even when names match. Class markers have `HAS_STANDARDIZATION_MARKER` edges rather than chemical-form semantics.

Eight Medium and eight Low candidates are retained under `review/pending_and_excluded.json`, outside active edges. Provenance sidecars preserve evidence records, source pointers and artifact hashes. IDs inside provenance/source metadata are archival references, not additional graph endpoints.

All quantities remain original label assertions. The 52 mL versus 0.52 mL conflict remains on the serving node; no per-unit or daily doses are derived. Reproduce from the project root with `supplementsrx_env/bin/python scripts/export_pilot_graph.py`.


In [9]:
from pathlib import Path
import json, hashlib
import pandas as pd
graph_root = Path.cwd()
if not (graph_root / 'data').is_dir():
    graph_root = graph_root.parent
graph_dir = graph_root / 'data/graph/pilot-graph-v1-094c1bd72c4dbab1'
graph_manifest = json.loads((graph_dir / 'manifest.json').read_text())
for name, expected_hash in graph_manifest['files'].items():
    assert hashlib.sha256((graph_dir / name).read_bytes()).hexdigest() == expected_hash
print('Counts:', graph_manifest['counts'])
print('Node types:', graph_manifest['node_types'])
print('Relationship types:', graph_manifest['edge_types'])
print('Validation:', graph_manifest['validation'])
graph_edges = pd.read_csv(graph_dir / 'relationships/relationships.csv', keep_default_na=False)
graph_nodes = pd.concat([pd.read_csv(f, keep_default_na=False) for f in sorted((graph_dir/'nodes').glob('*.csv'))], ignore_index=True)
assert graph_nodes.id.is_unique and graph_edges.id.is_unique
assert set(graph_edges.start_id) | set(graph_edges.end_id) <= set(graph_nodes.id)
print('\nActive mapping edges:')
print(graph_edges.loc[graph_edges.mapping_id != '', ['type','confidence','properties_json']].to_string(index=False))
print('\nFlagged serving:')
for row in graph_nodes.loc[graph_nodes.type == 'Serving'].itertuples():
    data = json.loads(row.properties_json)
    if any('numeric_text_serving_conflict' in i['reason_codes'] for i in data['unresolved_issues']):
        print(json.dumps(data, indent=2))


Counts: {'edges': 340, 'excluded_exact_matches': 8, 'mapping_edges': 24, 'nodes': 209, 'pending': 8}
Node types: {'ChemicalCompound': 2, 'IngredientOccurrence': 75, 'LabelAssertion': 103, 'ProductLabel': 7, 'Serving': 7, 'TerminologyConcept': 15}
Relationship types: {'ABOUT_OCCURRENCE': 75, 'DECLARES_OCCURRENCE': 75, 'DECLARES_SERVING': 7, 'DENOTES_CONCEPT': 18, 'EXACT_MATERIAL_MATCH': 6, 'HAS_ASSERTION': 103, 'HAS_DECLARED_FORM': 36, 'HAS_SERVING_BASIS': 16, 'HAS_STANDARDIZATION_MARKER': 4}
Validation: {'all_24_high_links_only': True, 'all_endpoints_resolve': True, 'canonical_merges': 0, 'derived_doses': 0, 'evidence_references_resolve': True, 'negative_checks_passed': 3, 'relationship_types_preserved': True, 'serving_conflict_retained': True, 'unique_ids': True}

Active mapping edges:
                type confidence                                                                                                                                                                           

### Export outcome and next step

209 nodes and 340 relationships were exported, including exactly 24 mapping links. The build validates unique IDs, endpoint integrity, mapping eligibility, relationship preservation and evidence references. Three negative checks confirm that dangling endpoints, Low-confidence mapping activation and concept-to-exact-match promotion fail validation. Repeating the exporter produces identical bytes.

Next, inspect product-to-ingredient-to-concept paths and a few POC questions against these tables before expanding the export. No database import, full-dataset expansion, product merging or model training has occurred.


## POC question checks against actual graph paths

**Current graph: `data/graph/pilot-graph-v1-d6d19a044338fcf0/`.** This supersedes the earlier `094c1bd72c4dbab1` build, which remains preserved. Path inspection found eight nested blend ingredients incorrectly exported as forms. These now use `HAS_DECLARED_COMPONENT`; explicit forms and standardization markers retain their distinct relationships. Counts remain 209 nodes, 340 edges and 24 mapping links.

These checks traverse the exported relationships, retrieve original assertion fields and preserve evidence pointers. They cover Q2, Q3, Q8 and the available portion of Q10 from the original question matrix, plus two relationship-semantics checks. The BPI comparison converts only plain mass units at an identical serving basis; it is not a dose recommendation, graph mutation or product merge.

Run `supplementsrx_env/bin/python scripts/export_pilot_graph.py` followed by `supplementsrx_env/bin/python scripts/inspect_pilot_paths.py` to reproduce. Full query results, original statements, paths and citations are saved in the path-review report.


In [10]:
from pathlib import Path
import json, hashlib
path_root = Path.cwd()
if not (path_root / 'data').is_dir():
    path_root = path_root.parent
path_report = json.loads((path_root / 'data/graph/pilot-path-review-v1-1339bd6889526e71/report.json').read_text())
current_graph = path_root / path_report['graph']
assert hashlib.sha256((current_graph / 'manifest.json').read_bytes()).hexdigest() == path_report['manifest_sha256']
print('Current graph:', path_report['graph'])
print('Summary:', path_report['summary'])
for case in path_report['cases']:
    print('\n' + case['id'] + ': ' + case['question'])
    print('Status:', case['status'])
    result = case['result']
    if isinstance(result, dict):
        print(result.get('answer', json.dumps(result, ensure_ascii=False)))
    else:
        for row in result:
            print('Label', row['label'], row['name'], 'percent:', row['percent'])
    if case['id'] == 'Q2':
        for row in result['records']:
            print('Label', row['label'], 'original:', row['original_quantity'], 'comparison mg:', row['comparison_mg'])
    if case['id'] == 'Q3':
        for row in result['labels']:
            print('Label', row['label'], 'other ingredients:', row['other_ingredients'])
            for statement in row['statements']:
                print('  Statement:', statement['fields'])
    if case['id'] == 'Q10':
        print('Original serving:', result['fields'])
print('\nExample source citations for Q2:')
print(json.dumps(path_report['cases'][0]['result']['records'][0]['citations'], indent=2))


Current graph: data/graph/pilot-graph-v1-d6d19a044338fcf0
Summary: {'answered': 4, 'answered_with_limit': 2, 'out_of_scope': 1}

Q2: Do the two BPI labels declare equal creatine mass per two capsules?
Status: answered
Yes: 1500 mg and 1.5 g are equivalent at the same two-capsule basis. Labels remain distinct; this is not a recommended dose or proof of duplicate formulations.
Label 25731 original: {'dailyValueTargetGroup': [{'footnote': 'Daily Value not established', 'name': 'Adults and children 4 or more years of age', 'operator': None, 'percent': None}], 'operator': '=', 'quantity': 1500, 'servingSizeOrder': 1, 'servingSizeQuantity': 2, 'servingSizeUnit': 'Capsule(s)', 'unit': 'mg'} comparison mg: 1500
Label 43261 original: {'dailyValueTargetGroup': [{'footnote': 'Daily Value not established', 'name': 'Adults and children 4 or more years of age', 'operator': None, 'percent': None}], 'operator': '=', 'quantity': 1.5, 'servingSizeOrder': 1, 'servingSizeQuantity': 2, 'servingSizeUnit': '

### Interpretation

- **Q2:** Both BPI labels declare equivalent creatine mass per two capsules (1500 mg and 1.5 g). They remain separate labels.
- **Q3:** Their other-ingredient lists differ. The graph retrieves each label's own statements rather than combining them.
- **Q8:** Creatine HCl reaches PubChem CID 134732 through exact-form links. The incompatible RxNorm candidates are not active links. Lack of a MeSH link here does not establish that MeSH has no suitable concept.
- **Q10:** Numeric 52 mL and note text 0.52 ml remain visible with source pointers. The graph cannot resolve the correct value or cause; the second comparison label is outside this pilot.
- **Additional checks:** Withanolides/Alkaloids remain class markers. Eight nested ingredients now use component edges.

**Coverage still missing:** Q1's Jarrow labels are absent; Q4–Q7 require NatMed graph data; Q9 requires Reddit graph data. Do not present these six passing path checks as full POC coverage or measured precision/recall.

Next, extend the graph to the missing question-relevant records and source types in bounded batches, keeping the same provenance and validation checks. A database import can follow inspection of the export format; no database or model training is required for these local checks.


## NatMed extension — one pending Ashwagandha safety annotation

**Current extended graph:** `data/graph/pilot-natmed-v1-22d3e423dc43f4fd/`. The prior DSLD/terminology graph is preserved unchanged. This build adds five nodes: monograph, monograph subject, pending clinical claim, source excerpt and captured reference document. There are 214 nodes and 349 relationships in total.

The selected annotation combines oral and topical contexts. Its original excerpt and unknown structured dose/population/formulation fields remain intact. It is not a complete Ashwagandha safety summary, a product safety determination or a dose recommendation. Clinical review remains **pending**.

### Ingredient-link review

Both DSLD Ashwagandha blends explicitly name the ingredient group Ashwagandha and botanical taxon *Withania somnifera*. The saved monograph identifies the same botanical subject. This supports two High-confidence `RELATED_BOTANICAL_SUBJECT` links at that limited scope, backed by the original label evidence and an anchored scientific-name section. The label's root extract/root powder blend is not equated with the monograph's unspecified formulation. No product-to-safety assertion or exact-material identity link is created.

The retrieval path is label → ingredient occurrence → related monograph subject ← pending claim → original excerpt → monograph. `ABOUT_SUBJECT` is traversed in reverse when locating claims. Mapping confidence and clinical approval remain separate.

The excerpt cites 23 reference numbers, all present in the captured bibliography. `CITES_ENTRIES` preserves those numbers; it does not assert that each study independently validates the annotation. Original capture completeness flags, reference text, source URL and evidence pointers remain available.

Reproduce with `supplementsrx_env/bin/python scripts/extend_graph_natmed.py` from the project root.


In [11]:
from pathlib import Path
import json, hashlib
import pandas as pd
natmed_root = Path.cwd()
if not (natmed_root / 'data').is_dir():
    natmed_root = natmed_root.parent
natmed_graph = natmed_root / 'data/graph/pilot-natmed-v1-22d3e423dc43f4fd'
natmed_manifest = json.loads((natmed_graph / 'manifest.json').read_text())
for filename, expected_hash in natmed_manifest['files'].items():
    assert hashlib.sha256((natmed_graph / filename).read_bytes()).hexdigest() == expected_hash
print('Graph counts:', natmed_manifest['counts'])
print('Validation:', natmed_manifest['validation'])
natmed_result = json.loads((natmed_graph / 'checks/natmed_retrieval.json').read_text())
print('\nQuestion:', natmed_result['question'])
print('Review status:', natmed_result['review_state'])
print('Clinical approval:', natmed_result['clinical_approval'])
print('Source:', natmed_result['source_url'])
print('Section:', natmed_result['section'])
print('\nOriginal saved excerpt:\n' + natmed_result['original_excerpt'])
print('\nPopulation:', natmed_result['population'])
print('Formulation:', natmed_result['formulation'])
print('Structured dose:', natmed_result['dose'])
print('Citation numbers:', natmed_result['citation_numbers'])
print('\nLimitations:')
for limitation in natmed_result['limitations']:
    print('-', limitation)
print('\nVerified paths:')
for path in natmed_result['paths']:
    print('Label:', path['label_id'])
    print(' -> '.join(path['traversal']))
    print(path['node_ids'])


Graph counts: {'cited_bibliography_entries': 23, 'edges': 349, 'nodes': 214, 'pending_clinical_claims': 1, 'related_subject_links': 2}
Validation: {'cited_numbers_in_bibliography': True, 'clinical_approvals': 0, 'endpoints_resolve': True, 'exact_material_links_unchanged': True, 'excerpt_offsets_match': True, 'no_product_safety_edges': True, 'original_graph_rows_unchanged': True, 'pending_review_preserved': True, 'safety_rating_span_matches': True, 'subject_links': 2, 'unknown_dose_and_population_preserved': True}

Question: What does the selected saved Ashwagandha safety annotation say, with its source and limitations?
Review status: pending
Clinical approval: False
Source: https://naturalmedicines.therapeuticresearch.com/IngredientsTherapiesMonographs/Pro?id=953
Section: safety

Original saved excerpt:
POSSIBLY SAFE when used orally and appropriately, short-term. Ashwagandha has been used with apparent safety in doses of up to 1250 mg daily for up to 6 months (3710, 11301, 19271, 9064

In [12]:
# Check the graph representation, rather than only the saved retrieval summary.
natmed_edges = pd.read_csv(natmed_graph / 'relationships/relationships.csv', keep_default_na=False)
natmed_claims = pd.read_csv(natmed_graph / 'nodes/ClinicalClaim.csv', keep_default_na=False)
assert len(natmed_claims) == 1
claim_data = json.loads(natmed_claims.iloc[0]['properties_json'])
assert claim_data['source_review']['state'] == 'pending'
assert claim_data['clinical_approval'] is False
assert all(v is None for v in claim_data['original_fields']['dose'].values())
assert claim_data['original_fields']['population'] is None
assert (natmed_edges.type == 'EXACT_MATERIAL_MATCH').sum() == 6
assert (natmed_edges.type == 'RELATED_BOTANICAL_SUBJECT').sum() == 2
assert (natmed_edges.type == 'HAS_PENDING_CLAIM').sum() == 1
print('Pending status, unknown dose/population and limited botanical-subject links verified.')
print(natmed_edges.loc[natmed_edges.type == 'RELATED_BOTANICAL_SUBJECT', ['type','confidence','properties_json']].to_string(index=False))


Pending status, unknown dose/population and limited botanical-subject links verified.
                     type confidence                                                                                                                                                                                                                                                                                                                                                                                                        properties_json
RELATED_BOTANICAL_SUBJECT       High {"clinical_approval": false, "reason": "DSLD ingredient group Ashwagandha and explicit Withania somnifera agree with monograph title and scientific-name section.", "rule": "NATMED-SUBJECT-01", "scope": "Shared botanical subject only; not exact formulation, product safety or dose applicability", "source_formulation": "Ashwagandha root extract & Ashwagandha root powder Blend", "source_role": "ingredient"}
RELATED_BOTANICAL_SUBJECT 

### Outcome and scope remaining

The first NatMed connection is now present and source-traceable. Original graph rows are preserved; excerpt offsets and the rating span match the saved source; graph endpoints and evidence references resolve; all 23 inline citation numbers appear in the bibliography. Repeating the exporter produces identical bytes.

This demonstrates **attributed retrieval of one pending safety annotation**, not validated clinical advice or complete Q5 coverage. Other safety warnings, interactions, effectiveness claims and dosing contexts still require their own bounded inclusion and checks. Reddit remains outside the graph. Raw data and `read.md` are unchanged.


## Reddit extension and source-separated retrieval

**Current graph:** `data/graph/pilot-reddit-v1-4029ca4e6dc2eae3/` — 225 nodes and 365 relationships. All six sources are now represented, with only bounded pilot coverage. Earlier graph builds remain unchanged.

Three Reddit comments retain their original text, URLs, timestamps, mention scope and parent/thread identifiers. They are `user_report` discussion/opinion records, not clinical claims. No personal-use, health-outcome or causality assertions are extracted.

- One comment explicitly mentions Creatine, but discusses packaging/scoops. Its `MENTIONS_TOPIC` edge has verified text spans; High confidence means confidence in the text mention only.
- Four topic associations across the other two comments are `INHERITED_THREAD_TOPIC`, preserved as upstream context and excluded from direct-mention results. The missing parent content has not independently verified those topics.
- Three `ExternalRecordReference` nodes preserve referenced post IDs with `content_loaded=false`. They are not fabricated Reddit post records. The unknown parent remains null; no reply edge is invented.
- Five `SupplementTopic` nodes support navigation, not chemical identity. Explicit DSLD ingredient-group topics and the saved NatMed topic provide source-separated retrieval paths. Reddit comments are not equated with chemical compounds or product-specific claims.

Reproduce with `supplementsrx_env/bin/python scripts/extend_graph_reddit.py`. The report under `checks/source_separated_retrieval.json` contains query results, source roles, citations and paths.


In [13]:
from pathlib import Path
import json, hashlib
import pandas as pd
reddit_root = Path.cwd()
if not (reddit_root / 'data').is_dir():
    reddit_root = reddit_root.parent
reddit_graph = reddit_root / 'data/graph/pilot-reddit-v1-4029ca4e6dc2eae3'
reddit_manifest = json.loads((reddit_graph / 'manifest.json').read_text())
for filename, expected in reddit_manifest['files'].items():
    assert hashlib.sha256((reddit_graph / filename).read_bytes()).hexdigest() == expected
print('Counts:', reddit_manifest['counts'])
print('Validation:', reddit_manifest['validation'])
source_queries = json.loads((reddit_graph / 'checks/source_separated_retrieval.json').read_text())
print('\nQuestion:', source_queries['question'])
for result in source_queries['queries']:
    print('\nTOPIC:', result['topic'])
    for role in ['product_labels', 'natmed_source_statements', 'reddit_direct_discussion', 'reddit_context_only_excluded_from_direct_results']:
        print('\n' + role + ':')
        for row in result[role]:
            print(json.dumps(row, indent=2, ensure_ascii=False))
        if not result[role]:
            print('No selected records for this evidence role; not a statement about the full source.')
    print('Coverage gaps:', result['gaps'])
print('\nLimit:', source_queries['limitation'])


Counts: {'context_only_associations': 4, 'direct_mentions': 1, 'edges': 365, 'external_reference_nodes': 3, 'nodes': 225, 'reddit_comments': 3, 'topic_nodes': 5}
Validation: {'clinical_approvals': 0, 'direct_text_spans_verified': True, 'endpoints_resolve': True, 'evidence_references_resolve': True, 'inherited_context_not_promoted': True, 'missing_parent_not_invented': True, 'original_rows_unchanged': True, 'raw_text_verified': True, 'source_separated_queries_passed': 2, 'unique_ids': True}

Question: What saved product labels, NatMed statements and Reddit discussions are available for Creatine and Ashwagandha?

TOPIC: Creatine

product_labels:
{
  "evidence_ids": [
    "evidence:a2b9c7cca0df5b3199ecfb802577553f9016915e51439a99dfcb77a6130663e8"
  ],
  "evidence_role": "label_information",
  "ingredient": "Creatine",
  "label_id": "43261",
  "path": [
    "snapshot:fc5ab272cc2b728e39f87525ac8b984841b6b2abb8d8913564315769e7b1106e",
    "occurrence:016e3b539ae46c6ce060e745c3e2c081b24dc924a

In [14]:
reddit_edges = pd.read_csv(reddit_graph / 'relationships/relationships.csv', keep_default_na=False)
reddit_comments = pd.read_csv(reddit_graph / 'nodes/RedditComment.csv', keep_default_na=False)
assert len(reddit_comments) == 3
assert (reddit_edges.type == 'MENTIONS_TOPIC').sum() == 1
assert (reddit_edges.type == 'INHERITED_THREAD_TOPIC').sum() == 4
assert (reddit_edges.type == 'REPLY_TO_REFERENCE').sum() == 2
for row in reddit_comments.itertuples():
    comment = json.loads(row.properties_json)
    print('\n', comment['attributes']['native_id'], comment['attributes']['mention_scope'])
    print(comment['attributes']['text'])
    print('Source:', comment['attributes']['source_url'])
    print('Parent:', comment['attributes']['parent_id'], 'Thread:', comment['attributes']['thread_id'])
    assert comment['clinical_approval'] is False
    if comment['attributes']['parent_id'] is None:
        assert not ((reddit_edges.start_id == row.id) & (reddit_edges.type == 'REPLY_TO_REFERENCE')).any()
print('\nThree original comments retained; inherited topics and unknown parent handled separately.')



 t1_19rg1ztatzi8z thread_context
I vaporise CBN hash and it’s wonderful
Source: https://old.reddit.com/r/Supplements/comments/1xbme8k/legalhigh_supplements/19rg1ztatzi8z/
Parent: t3_1xbme8k Thread: t3_1xbme8k

 t1_19rgc7uzlnucg own_text
If the problem you’re trying to solve is the nuisance of powder covered fingers, I’d say scrap the “revamping” of a free utensil included in whey or creatine products, and focus on something like perfectly shaped tongs or a small sieve using the same material used to create ur scoop. I just don’t think it’s annoying enough for someone to purchase a completely different product when the product is included in whey or creatine, and when a simple solution is to keep the scoops out of the packaging. Even better then, maybe design something that can universally fit around or on packaging so people can fasten their plastic scoop to it. Just some thoughts but keep designing and creating!
Source: https://old.reddit.com/r/Supplements/comments/1xbmiz3/designed_a

### Findings and remaining coverage

The two source-separated queries pass:

| Topic | DSLD | NatMed | Reddit direct discussion |
|---|---|---|---|
| Creatine | Two BPI labels | None selected | One packaging discussion |
| Ashwagandha | Two labels | One pending safety annotation | None selected |

A context-only Creatine association is available separately but does not enter direct-discussion results. No selected supplement has all three evidence roles. This is a coverage gap, not permission to manufacture a common-topic result or claim personal experiences.

Validation checks unique IDs, graph endpoints, unchanged prior rows, original text, mention spans, source evidence references and absence of invented parent links. Raw data and `read.md` are unchanged; no Neo4j import, clinical approvals or training occurred.

Next: choose one concrete POC question/topic and expand its evidence coverage in a bounded batch, or inspect the current graph before database import. Source-specific gaps should remain explicit in either case.


## Creatine NatMed coverage — bounded selection with confidence rules

**Current graph:** `data/graph/creatine-natmed-v1-f174e5e28b21603a/` — 249 nodes and 418 relationships. Previous builds and raw files are preserved.

The saved Creatine dataset contains 76 annotations: 50 effectiveness, 15 adverse-effect, 7 safety, 2 dose and 2 interaction annotations. This batch selects **10**: the athletic-performance effectiveness annotation, all seven safety annotations (including population/condition cautions), and both adult dosing annotations. The 66 unselected records are outside this batch, not classified as unreliable.

| Confidence | Technical criteria | Export action |
|---|---|---|
| High | Compatible general ingredient subject; valid schema; exact source binding, excerpt and populated field-support spans | Include as pending, attributed content |
| Medium | Missing positive field support | Hold for review |
| Low | Conflicting identity, source, schema or spans | Exclude from active export |

All 10 selected annotations meet High **technical** confidence. Confidence is not clinical approval, completeness or certainty of effectiveness/safety. All remain pending. Original source ratings and qualifications are retained independently from the technical confidence label.

The monograph connects to the Creatine navigation topic and to the two general DSLD Creatine ingredient occurrences via `RELATED_INGREDIENT_SUBJECT`. No exact-form, branded-product or dose-applicability connection is inferred. The athletic-performance passage refers to monohydrate and retains its full qualification text; it must not be generalized to HCl products.

The two dosing annotations share one source excerpt but describe loading and maintenance fields separately. Both claims are preserved, while the identical source excerpt is stored once. Unknown population, formulation and indication fields remain unknown. Full warnings and formulation context are also preserved without converting them into approved advice.

Reproduce with `supplementsrx_env/bin/python scripts/expand_creatine_natmed.py`. See `review/creatine_selection.json` for selected and out-of-scope records, rules and reasons.


In [15]:
from pathlib import Path
import json, hashlib
import pandas as pd
creatine_root = Path.cwd()
if not (creatine_root / 'data').is_dir():
    creatine_root = creatine_root.parent
creatine_graph = creatine_root / 'data/graph/creatine-natmed-v1-f174e5e28b21603a'
creatine_manifest = json.loads((creatine_graph / 'manifest.json').read_text())
for filename, expected in creatine_manifest['files'].items():
    assert hashlib.sha256((creatine_graph / filename).read_bytes()).hexdigest() == expected
print('Counts:', creatine_manifest['counts'])
print('Validation:', creatine_manifest['validation'])
creatine_selection = json.loads((creatine_graph / 'review/creatine_selection.json').read_text())
print('Available:', creatine_selection['available_by_type'])
print(pd.DataFrame(creatine_selection['selected'])[['claim_type','population','indication','confidence','review_state','clinical_approval']].fillna('<unknown>').to_string(index=False))
print('Outside this batch:', len(creatine_selection['not_selected']))
creatine_query_report = json.loads((creatine_graph / 'checks/source_separated_retrieval.json').read_text())
creatine_query = next(q for q in creatine_query_report['queries'] if q['topic'] == 'Creatine')
print('\nSource-role counts:', {k: len(creatine_query[k]) for k in ['product_labels','natmed_source_statements','reddit_direct_discussion','reddit_context_only_excluded_from_direct_results']})
print('Limitation:', creatine_query_report['limitation'])
assert len(creatine_query['natmed_source_statements']) == 10
assert all(r['review_state'] == 'pending' and r['clinical_approval'] is False for r in creatine_query['natmed_source_statements'])


Counts: {'added_claims': 10, 'edges': 418, 'nodes': 249, 'selected_by_type': {'dose': 2, 'effectiveness': 1, 'safety': 7}, 'selected_confidence': {'High': 10}}
Validation: {'all_selected_citations_in_bibliography': True, 'clinical_approvals': 0, 'creatine_three_source_roles': True, 'endpoints_resolve': True, 'negative_checks_passed': 3, 'original_graph_rows_preserved': True, 'pending_status_preserved': True, 'selected_source_and_field_spans_verified': True, 'unknown_fields_preserved': True}
Available: {'adverse_effect': 15, 'dose': 2, 'effectiveness': 50, 'interaction': 2, 'safety': 7}
   claim_type              population            indication confidence review_state  clinical_approval
       safety               <unknown>             <unknown>       High      pending              False
       safety               <unknown>             <unknown>       High      pending              False
       safety                CHILDREN             <unknown>       High      pending              F

In [16]:
# Original attributed passages, retained alongside structured fields.
for claim in creatine_query['natmed_source_statements']:
    print('\nCLAIM:', claim['claim_id'])
    print('Type:', claim['claim_type'], '| Technical confidence:', claim['technical_confidence'], '| Review:', claim['review_state'])
    print('Population:', claim['population'], '| Indication:', claim['indication'])
    print('Source rating:', claim['safety_rating'] or claim['effectiveness_rating'])
    print('Ingredient context:', claim['ingredient'])
    print('Source:', claim['source_url'])
    print('Original excerpt:\n' + claim['original_excerpt'])
    if claim['claim_type'] == 'dose':
        print('Structured dose fields:', claim['dose'])
    print('Citation numbers:', claim['citation_numbers'])
    print('Evidence references:', claim['evidence_ids'])



CLAIM: claim:88cf9029a0198b6bea27d2d1d2da03c5442592b33d5797117481f40cd41d4b3c
Type: safety | Technical confidence: High | Review: pending
Population: None | Indication: None
Source rating: LIKELY SAFE
Ingredient context: {'formulation': None, 'name': 'Creatine', 'strain': None}
Source: https://naturalmedicines.therapeuticresearch.com/IngredientsTherapiesMonographs/Pro?id=873
Original excerpt:
LIKELY SAFE when used orally and appropriately, short-term. Creatine supplementation appears to be safe when used at loading doses of up to 25 grams daily or 0.3 grams/kg daily for up to 14 days in healthy adults (1367, 2100, 2101, 3996, 4569, 10064, 15354, 15520, 46570, 46587)(46673, 46688, 46719, 46753, 46801, 103278, 103279, 108336). Creatine supplementation also appears to be safe when used at maintenance doses of 4-5 grams daily for up to 18 months (2101, 4578, 15353, 15354, 15520, 46587, 46673, 46690, 46753, 46838, 102164, 103278, 108336).
Citation numbers: ['1367', '2100', '2101', '3996', 

### Result and limits

The Creatine query now has all three evidence roles: **two DSLD labels, ten pending NatMed annotations and one direct Reddit discussion**. The Reddit record concerns packaging, not a reported health effect. A context-only Reddit association remains excluded from direct-mention results.

Checks verify exact source/excerpt/field support, source citation presence in the bibliography, graph endpoint integrity, unchanged prior graph rows and pending review status. Negative fixtures reject an incompatible ingredient or corrupted excerpt and hold missing field support. Repeated execution produces identical artifacts.

This does not complete Creatine clinical coverage: adverse effects, interactions and other effectiveness contexts are outside the selected batch. The seven safety annotations are not a substitute for the whole monograph. High technical confidence never implies that an ingredient or branded product is safe for a particular person.


## Sample user-facing answer review — relevance, scope and citations

This is a **fixed, curated POC example**, not a general question-answering or ranking engine. It uses only the saved graph; no live medical guidance is generated. The graph itself is unchanged.

**Question:** What do the saved sources say about creatine for athletic performance, and what forms and amounts do the two BPI Creatine Alkaline labels declare?

The answer selects one effectiveness annotation and the two label records. It excludes nine safety/dosing annotations because they are outside this question, the direct Reddit packaging discussion because it is irrelevant to effectiveness, and the inherited Reddit association because it is not a direct statement. Exclusion here concerns relevance, not source quality or clinical truth.

Clinical-source content stays attributed and pending. Monohydrate scope, tentative effectiveness rating, uncertainty and conflicting findings are preserved. Label amounts are not represented as recommended doses, and the source passage is not treated as proof that a particular BPI formulation works.

Reproduce with `supplementsrx_env/bin/python scripts/review_creatine_answer.py`. The saved JSON includes the draft, sentence-group evidence references, graph paths, exclusion reasons and review limitations.


In [17]:
from pathlib import Path
import json, hashlib
answer_root = Path.cwd()
if not (answer_root / 'data').is_dir():
    answer_root = answer_root.parent
answer_review = json.loads((answer_root / 'data/graph/creatine-answer-review-v1-d8909281d959c7c8/report.json').read_text())
assert hashlib.sha256((answer_root / answer_review['graph'] / 'manifest.json').read_bytes()).hexdigest() == answer_review['graph_manifest_sha256']
print('QUESTION:', answer_review['question'])
for section in answer_review['answer_sections']:
    print('\n' + section['text'])
    if section.get('source_url'):
        print('Source:', section['source_url'], '| Review:', section['review_state'])
    if section.get('label_records'):
        print('Sources: saved DSLD labels', ', '.join(r['label_id'] for r in section['label_records']))
    print('Evidence references:', section['evidence_ids'])
print('\nReview:', answer_review['review'])
print('Counts:', answer_review['counts'])


QUESTION: What do the saved sources say about creatine for athletic performance, and what forms and amounts do the two BPI Creatine Alkaline labels declare?

The saved NatMed monograph rates athletic performance as POSSIBLY EFFECTIVE. Its selected passage describes possible modest benefits of oral creatine monohydrate for rowing, jumping and soccer, while benefits for sprinting, cycling, swimming and tennis are uncertain. It also reports conflicting findings. This annotation is pending clinical review.
Source: https://naturalmedicines.therapeuticresearch.com/IngredientsTherapiesMonographs/Pro?id=873 | Review: pending
Evidence references: ['creatine-expansion-evidence:d3665e07843e18bc7f67ab87f32f2748d374b51c4a610308907d677f424bad57', 'creatine-expansion-evidence:9759746f9bd0e96db81392ef6b07381ca64257bfab436d803e6f72a9cddbd7a7', 'creatine-expansion-evidence:aa869dfa727e50561e9db2360400a848c5ad0b638949d5eaad328c36481555d5', 'creatine-expansion-evidence:c2ae89a2645b358fda1cf7e94513b47f0882

In [18]:
print('EXCLUDED RECORDS AND REASONS')
for record in answer_review['excluded_records']:
    print(json.dumps(record, ensure_ascii=False))
print('\nSOURCE LOCATORS AND HASHES')
for section, references in answer_review['source_references'].items():
    print('\nSection:', section)
    for reference in references:
        print(json.dumps(reference, ensure_ascii=False))
assert answer_review['counts']['included_natmed_claims'] == 1
assert answer_review['counts']['included_label_records'] == 2
assert answer_review['counts']['included_reddit_records'] == 0
assert answer_review['review']['clinical_approval'] is False


EXCLUDED RECORDS AND REASONS
{"reason": "Safety or dosing context is outside this effectiveness-and-label-composition question; not judged unreliable", "record_id": "claim:88cf9029a0198b6bea27d2d1d2da03c5442592b33d5797117481f40cd41d4b3c", "source": "NatMed"}
{"reason": "Safety or dosing context is outside this effectiveness-and-label-composition question; not judged unreliable", "record_id": "claim:eebb885776a41b2ded96c9f777d2e48d3f9e332abe785834e7f20f0071e50b65", "source": "NatMed"}
{"reason": "Safety or dosing context is outside this effectiveness-and-label-composition question; not judged unreliable", "record_id": "claim:6360244495b4dfad328a9ca9f51325cb6275dea749b4d3364d1b4501f30b90ec", "source": "NatMed"}
{"reason": "Safety or dosing context is outside this effectiveness-and-label-composition question; not judged unreliable", "record_id": "claim:c4bcd4bcd3d166d0a751b8e72ef1a0e12a2c0a67a41f3be1c476a52c33811f3d", "source": "NatMed"}
{"reason": "Safety or dosing context is outside thi

### Review outcome

Source paths, original quantity/serving bases, forms and evidence references pass the checks. Negative guards reject a falsely approved annotation, a missing evidence reference and inclusion of irrelevant Reddit content. Assistant semantic review confirms that the paraphrase retains source qualifications; automated checks are not proof of medical correctness.

The useful finding is that **High technical confidence does not determine answer relevance**. The answer must select records appropriate to the question, while preserving source role and review status. One passing example does not establish general answer quality, precision or recall.

Next: add a few contrasting fixed questions (for example, label comparison, a source safety question and an unanswerable query) to check that relevance and abstention remain correct before expanding the dataset or building a general conversational layer.


## Contrasting answer checks — comparison, safety retrieval and insufficient evidence

Three additional fixed examples test whether the selected source role and answer boundary change with the question. Together with the earlier athletic-performance example, there are now four curated answer scenarios. These are not a general conversational system or a measured precision/recall evaluation.

1. **Label comparison:** equal creatine mass per two capsules, but differing other-ingredient lists. No product merge or dose recommendation.
2. **Source safety question:** retrieve all seven selected Creatine safety passages with their source ratings, population fields, qualifications and pending status. Do not collapse them into a universal safety rating.
3. **Product-effectiveness comparison:** report insufficient evidence. General ingredient research, equal label amounts and packaging discussion cannot determine which BPI product works better.

High technical confidence remains an eligibility rule. Relevance determines which eligible records answer a particular question. Missing product-comparison evidence is a limitation of this graph, not proof that no such evidence exists anywhere.

Reproduce with `supplementsrx_env/bin/python scripts/review_contrasting_answers.py`. Source evidence, exclusion reasons, coverage checks and validations are saved in the accompanying JSON report. Graph data is unchanged.


In [19]:
from pathlib import Path
import json, hashlib
contrast_root = Path.cwd()
if not (contrast_root / 'data').is_dir():
    contrast_root = contrast_root.parent
contrast_review = json.loads((contrast_root / 'data/graph/contrasting-answer-review-v1-e2c229e10e5ccb52/report.json').read_text())
assert hashlib.sha256((contrast_root / contrast_review['graph'] / 'manifest.json').read_bytes()).hexdigest() == contrast_review['graph_manifest_sha256']
for case in contrast_review['cases']:
    print('\nQUESTION:', case['question'])
    print('STATUS:', case['status'])
    print('ANSWER:', case['answer'])
    print('Included labels:', case.get('included_label_ids', []))
    print('Included claims:', case['included_claim_ids'])
    print('Excluded roles:', case.get('excluded_roles', []))
    print('Reason:', case.get('exclusion_reason'))
    if case['id'] == 'label_comparison':
        for record in case['source_records']:
            print('Label', record['label_id'], 'other ingredients:', record['other_ingredients'])
    if case['id'] == 'unanswerable':
        print('Coverage check:', case['coverage_check'])
print('\nValidation:', contrast_review['validation'])
print('Review limits:', contrast_review['review'])



QUESTION: Do the two saved BPI Creatine Alkaline labels have the same creatine amount and the same formulation?
STATUS: answered_with_limit
ANSWER: Both declare the same total creatine mass per two capsules: 1,500 mg on label 25731 and 1.5 g on label 43261. Both list creatine monohydrate and Creatine HCl, but their other-ingredient lists differ. Equal amounts do not establish identical formulations or justify merging these labels. These are saved label declarations, not recommended doses.
Included labels: ['25731', '43261']
Included claims: []
Excluded roles: ['NatMed clinical statements', 'Reddit discussion']
Reason: Neither is needed to compare the saved labels.
Label 25731 other ingredients: ['Dicalcium Phosphate', 'Magnesium Stearate', 'Silica', 'pH-Sorb(TM) Buffering Technology']
Label 43261 other ingredients: ['Dicalcium Phosphate', 'FD&C Blue No. 1', 'Magnesium Stearate', 'Silica', 'Titanium Dioxide', 'pH-Sorb(TM) Buffering Technology']

QUESTION: What safety statements and pop

In [20]:
# Show each selected safety source separately, preserving the full qualifications.
safety_example = next(c for c in contrast_review['cases'] if c['id'] == 'source_safety')
for record in safety_example['source_records']:
    print('\nPopulation:', record['population'] if record['population'] is not None else '<unknown structured population>')
    print('Source rating:', record['source_rating'])
    print('Review:', record['review_state'], '| Clinical approval:', record['clinical_approval'])
    print('Source:', record['source_url'])
    print('Original saved excerpt:\n' + record['original_excerpt'])
    print('Evidence IDs:', record['evidence_ids'])
print('\nEvidence pointers for all answer sections:')
for case_id, references in contrast_review['source_references'].items():
    print(case_id, json.dumps(references, ensure_ascii=False))
assert len(safety_example['source_records']) == 7
assert all(r['review_state'] == 'pending' for r in safety_example['source_records'])



Population: <unknown structured population>
Source rating: LIKELY SAFE
Review: pending | Clinical approval: False
Source: https://naturalmedicines.therapeuticresearch.com/IngredientsTherapiesMonographs/Pro?id=873
Original saved excerpt:
LIKELY SAFE when used orally and appropriately, short-term. Creatine supplementation appears to be safe when used at loading doses of up to 25 grams daily or 0.3 grams/kg daily for up to 14 days in healthy adults (1367, 2100, 2101, 3996, 4569, 10064, 15354, 15520, 46570, 46587)(46673, 46688, 46719, 46753, 46801, 103278, 103279, 108336). Creatine supplementation also appears to be safe when used at maintenance doses of 4-5 grams daily for up to 18 months (2101, 4578, 15353, 15354, 15520, 46587, 46673, 46690, 46753, 46838, 102164, 103278, 108336).
Evidence IDs: ['creatine-expansion-evidence:3e19c0e4cc75db7ca6c3ad3bb557b420636ffee3c82bdfeab3e117c07218d122', 'creatine-expansion-evidence:8e02e68f8efa94c566eaa0796b3fd5b52531c8d019c5e82cdf48e27fb69bbeaf', 'cr

### Findings

All three fixed cases pass their structural/source checks. Four negative guards reject false product equivalence, an improperly approved safety annotation, an invented recommendation and a missing citation. Assistant semantic review confirms that the wording retains the selected evidence role and limitations; this is not automated medical validation.

The current examples demonstrate four behaviors: scoped effectiveness retrieval, label comparison without merging, qualified source safety retrieval and an explicit insufficient-evidence response. The graph remains at 249 nodes and 418 relationships. No raw files, source decisions or graph links were changed.

Next, review these sample answers with the project team. The remaining implementation choice is whether to build a small general retrieval layer on the validated graph or first expand coverage for additional POC topics; these curated examples alone do not establish readiness for clinical use.
